In [1]:
import pandas as pd
import joblib
from sklearn.metrics import classification_report

# PRODUCTION FUNCTION
def production(X_path, y_path):

    # Load saved model + feature list
    model = joblib.load("logreg_fake_profile_model.pkl")
    numeric_features = joblib.load("numeric_features.pkl")
    id_col = "profile_id" 

    # Load new data
    df_X = pd.read_csv(X_path)

    # Feature Engineering
    df_fe = df_X.copy()

    df_fe["follow_ratio"] = df_fe["followers_count"] / (df_fe["following_count"] + 1)
    df_fe["posts_per_day"] = df_fe["posts_count"] / (df_fe["account_age_days"] + 1)
    df_fe["recent_post_rate_30d"] = df_fe["posts_last_30_days"] / 30.0
    df_fe["repeated_caption_rate"] = df_fe["repeated_captions"] / (df_fe["recent_posts_sampled"] + 1)
    df_fe["night_post_rate"] = df_fe["night_posts"] / (df_fe["recent_posts_sampled"] + 1)
    df_fe["likes_per_post_20"] = df_fe["likes_last_20_posts"] / 20.0
    df_fe["comments_per_post_20"] = df_fe["comments_last_20_posts"] / 20.0
    df_fe["dm_reply_rate"] = df_fe["dm_replies_last_7_days"] / (df_fe["dm_sent_last_7_days"] + 1)
    df_fe["username_digit_density"] = df_fe["username_digit_count"] / (df_fe["username_length"] + 1)
    df_fe["links_per_post"] = df_fe["external_link_count"] / (df_fe["posts_count"] + 1)

    # Drop redundant columns
    df_fe = df_fe.drop(columns=[
        "posts_count",
        "likes_last_20_posts",
        "comments_last_20_posts",
        "username_digit_count",
        "username_length",
        "night_posts"
    ])

    # Select numeric features
    X_final = df_fe[numeric_features]

    # Predict
    pred = model.predict(X_final)

    # Load true labels
    df_Y = pd.read_csv(y_path)
    y_label = df_Y.columns[-1]
    df_y = df_Y[y_label]

    # Evaluation
    print(classification_report(df_y, pred))


production( 
    X_path='https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/prod-fake_profiles_unlabeled.csv',
    y_path='https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/final/prod-fake_profiles_labels.csv'
)

              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1582
           1       0.90      0.93      0.92       918

    accuracy                           0.94      2500
   macro avg       0.93      0.94      0.93      2500
weighted avg       0.94      0.94      0.94      2500

